![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/master/competitions/structural-break-real-time/assets/banner.webp)

In [ ]:
%pip install crunch-cli catboost --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time PASTE-FRESH-TOKEN-HERE

import math, os, time, gc
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import crunch
crunch_tools = crunch.load_notebook()

In [ ]:
from collections import deque

def _win_stats_blocks(zh, w):
    n = len(zh)
    if n < 2*w:
        return None
    step = max(w // 2, 1)
    lvars, kss, skews, ac1s, means = [], [], [], [], []
    qs = np.quantile(zh, np.linspace(0.1, 0.9, 9))
    for start in range(0, n - w + 1, step):
        seg = zh[start:start+w]
        m = seg.mean(); v = max(seg.var(), 1e-12)
        lvars.append(math.log(v))
        zn = (seg - m)/math.sqrt(v)
        skews.append(float((zn**3).mean()))
        a = seg - m
        ac1s.append(float(np.dot(a[1:], a[:-1]) / max(np.dot(a, a), 1e-12)))
        emp = np.searchsorted(np.sort(seg), qs, side="right") / w
        kss.append(float(np.max(np.abs(emp - np.linspace(0.1, 0.9, 9)))))
        means.append(m)
    def musd(v):
        v = np.asarray(v); return float(v.mean()), max(float(v.std()), 1e-6)
    return {"lvar": musd(lvars), "ks": musd(kss), "skew": musd(skews),
            "ac1": musd(ac1s), "mean": musd(means)}

def _fit_ar2(z):
    if len(z) < 50:
        return 0.0, 0.0, 1.0
    y = z[2:]; X = np.column_stack([z[1:-1], z[:-2]])
    XtX = X.T @ X + 1e-6 * np.eye(2)
    phi = np.linalg.solve(XtX, X.T @ y)
    resid = y - X @ phi
    return float(phi[0]), float(phi[1]), max(float(resid.var()), 1e-6)

class StreamingFeatures:
    def __init__(self, x_hist):
        x = np.asarray(x_hist, dtype=np.float64)
        self.mu_h = float(x.mean()); self.sd_h = max(float(x.std(ddof=1)), 1e-8)
        xc = x - self.mu_h
        den_h = max(np.dot(xc, xc), 1e-12)
        self.ac1_h = float(np.dot(xc[1:], xc[:-1]) / den_h)
        self.ac2_h = float(np.dot(xc[2:], xc[:-2]) / den_h)
        self.ac5_h = float(np.dot(xc[5:], xc[:-5]) / den_h)
        zh = xc / self.sd_h
        self.qs = np.quantile(zh, np.linspace(0.1, 0.9, 9))
        self.skew_h = float((zh**3).mean())
        self.kurt_h = float((zh**4).mean()) - 3.0
        self.cal100 = _win_stats_blocks(zh, 100)
        self.cal20 = _win_stats_blocks(zh, 20)
        self.t = 0; self.sum = 0.0; self.sumsq = 0.0
        self.cusum_pos = 0.0; self.cusum_neg = 0.0; self.n_tail = 0
        self.prev_z = None; self.ac1_num = 0.0; self.ac1_den = 0.0
        self.w20 = deque(); self.s20 = 0.0; self.q20 = 0.0
        self.w100 = deque(maxlen=100); self.s100 = 0.0; self.q100 = 0.0
        self.pz = [0.0]; self.pz2 = [0.0]; self.pz3 = [0.0]; self.pt = [0]
        self.phi1, self.phi2, self.rv = _fit_ar2(zh)
        self.z1 = zh[-1]; self.z2 = zh[-2] if len(zh) >= 2 else 0.0
        self.ll_cum = 0.0
        self.r_sum = 0.0; self.r_sq = 0.0; self.r_n = 0
        self.r_prev = None; self.r_ac_num = 0.0; self.r_ac_den = 0.0
        self.rw = deque(maxlen=50); self.rw_s = 0.0; self.rw_q = 0.0
        self.r_cusum = 0.0
        self.ew_var = 1.0; self.g_cusum = 0.0; self.g_ll = 0.0; self.g_n = 0
        self.u3 = 0.0; self.u4 = 3.0

    def update(self, x):
        self.t += 1
        z = (x - self.mu_h) / self.sd_h
        k = 0.5
        self.cusum_pos = max(0.0, self.cusum_pos + z - k)
        self.cusum_neg = max(0.0, self.cusum_neg - z - k)
        cusum = max(self.cusum_pos, self.cusum_neg)
        self.sum += x; self.sumsq += x * x
        if self.t > 1:
            m = self.sum / self.t
            var_o = max((self.sumsq - self.t * m * m) / (self.t - 1), 1e-12) / self.sd_h ** 2
        else:
            var_o = 1.0
        var_ratio = math.log(var_o)
        if abs(z) > 2.0: self.n_tail += 1
        tail_frac = self.n_tail / self.t
        if self.prev_z is not None:
            self.ac1_num += z * self.prev_z; self.ac1_den += z * z
        ac1_o = self.ac1_num / self.ac1_den if self.ac1_den > 1e-12 else 0.0
        self.prev_z = z

        self.w20.append(z); self.s20 += z; self.q20 += z * z
        if len(self.w20) > 20:
            old = self.w20.popleft(); self.s20 -= old; self.q20 -= old * old
        n20 = len(self.w20); m20 = self.s20 / n20
        v20 = max(self.q20 / n20 - m20 * m20, 1e-12)

        if len(self.w100) == 100:
            old = self.w100[0]; self.s100 -= old; self.q100 -= old * old
        self.w100.append(z); self.s100 += z; self.q100 += z * z
        n100 = len(self.w100); m100 = self.s100 / n100
        v100 = max(self.q100 / n100 - m100 * m100, 1e-12)
        arr = np.asarray(self.w100)
        zn = (arr - m100) / math.sqrt(v100)
        skew_raw = float((zn**3).mean())
        w100_skew = skew_raw - self.skew_h
        w100_kurt = float((zn**4).mean()) - 3.0 - self.kurt_h
        a = arr - m100
        den = max(np.dot(a, a), 1e-12)
        ac1_100 = float(np.dot(a[1:], a[:-1]) / den) if n100 >= 5 else self.ac1_h
        w100_ac1 = ac1_100 - self.ac1_h
        w100_ac2 = (float(np.dot(a[2:], a[:-2]) / den) if n100 >= 7 else self.ac2_h) - self.ac2_h
        w100_ac5 = (float(np.dot(a[5:], a[:-5]) / den) if n100 >= 12 else self.ac5_h) - self.ac5_h
        emp = np.searchsorted(np.sort(arr), self.qs, side="right") / n100
        ks100 = float(np.max(np.abs(emp - np.linspace(0.1, 0.9, 9))))
        signs = np.sign(arr)
        flips = float(np.mean(signs[1:] * signs[:-1] < 0)) if n100 >= 3 else 0.5

        c = self.cal100
        if c is not None and n100 >= 50:
            cal_lvar = (math.log(v100) - c["lvar"][0]) / c["lvar"][1]
            cal_ks   = (ks100 - c["ks"][0]) / c["ks"][1]
            cal_skew = (skew_raw - c["skew"][0]) / c["skew"][1]
            cal_ac1  = (ac1_100 - c["ac1"][0]) / c["ac1"][1]
        else:
            cal_lvar = cal_ks = cal_skew = cal_ac1 = 0.0
        c2 = self.cal20
        if c2 is not None and n20 >= 10:
            cal_mean20 = (m20 - c2["mean"][0]) / c2["mean"][1]
        else:
            cal_mean20 = 0.0

        self.pz.append(self.pz[-1] + z)
        self.pz2.append(self.pz2[-1] + z * z)
        self.pz3.append(self.pz3[-1] + z ** 3)
        self.pt.append(self.pt[-1] + (1 if abs(z) > 2 else 0))
        t = self.t
        scan_mean = scan_lvar = scan_skew = scan_tail = 0.0
        if t >= 8:
            for f in (0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85):
                s = max(2, int(t * f))
                if s >= t - 1: continue
                n1, n2 = s, t - s
                m1 = self.pz[s] / n1; m2 = (self.pz[t] - self.pz[s]) / n2
                v1 = max(self.pz2[s] / n1 - m1 * m1, 1e-12)
                v2 = max((self.pz2[t] - self.pz2[s]) / n2 - m2 * m2, 1e-12)
                dmean = abs(m2 - m1) / math.sqrt(v1 / n1 + v2 / n2)
                dlvar = abs(math.log(v2 / v1))
                mu3_1 = self.pz3[s] / n1 - 3 * m1 * v1 - m1 ** 3
                mu3_2 = (self.pz3[t] - self.pz3[s]) / n2 - 3 * m2 * v2 - m2 ** 3
                dskew = abs(mu3_2 / v2 ** 1.5 - mu3_1 / v1 ** 1.5)
                dtail = abs((self.pt[t] - self.pt[s]) / n2 - self.pt[s] / n1)
                if dmean > scan_mean: scan_mean = dmean
                if dlvar > scan_lvar: scan_lvar = dlvar
                if dskew > scan_skew: scan_skew = dskew
                if dtail > scan_tail: scan_tail = dtail

        pred = self.phi1 * self.z1 + self.phi2 * self.z2
        u = (z - pred) / math.sqrt(self.rv)
        self.z2 = self.z1; self.z1 = z
        self.ll_cum += 0.5 * (u * u - 1.0)
        self.r_n += 1; self.r_sum += u; self.r_sq += u * u
        r_var = self.r_sq / self.r_n - (self.r_sum / self.r_n) ** 2 if self.r_n > 1 else 1.0
        if self.r_prev is not None:
            self.r_ac_num += u * self.r_prev; self.r_ac_den += u * u
        r_ac = self.r_ac_num / self.r_ac_den if self.r_ac_den > 1e-12 else 0.0
        self.r_prev = u
        if len(self.rw) == 50:
            old = self.rw[0]; self.rw_s -= old; self.rw_q -= old * old
        self.rw.append(u); self.rw_s += u; self.rw_q += u * u
        nw = len(self.rw); mw = self.rw_s / nw
        vw = max(self.rw_q / nw - mw * mw, 1e-12)
        self.r_cusum = max(0.0, self.r_cusum + u - 0.5)
        ll_norm = self.ll_cum / math.sqrt(max(self.r_n, 1))

        g = u / math.sqrt(max(self.ew_var, 1e-6))
        self.ew_var = 0.94 * self.ew_var + 0.06 * u * u
        self.g_n += 1
        self.g_ll += 0.5 * (g * g - 1.0)
        self.g_cusum = max(0.0, self.g_cusum + abs(g) - 0.8)
        self.u3 = 0.97 * self.u3 + 0.03 * (u ** 3)
        self.u4 = 0.97 * self.u4 + 0.03 * (u ** 4)
        g_ll_norm = self.g_ll / math.sqrt(max(self.g_n, 1))
        g_cusum_n = self.g_cusum / math.sqrt(max(self.g_n, 1))

        return (self.t, cusum, var_ratio, tail_frac, ac1_o - self.ac1_h, abs(z),
                m20 * math.sqrt(n20), math.log(v20),
                m100 * math.sqrt(n100), math.log(v100),
                w100_skew, w100_kurt, w100_ac1, ks100, flips,
                cal_lvar, cal_ks, cal_skew, cal_ac1, cal_mean20,
                w100_ac2, w100_ac5, scan_mean, scan_lvar, scan_skew, scan_tail,
                ll_norm, math.log(max(r_var, 1e-12)), r_ac,
                math.log(vw), mw * math.sqrt(nw), self.r_cusum,
                g_ll_norm, g_cusum_n, math.log(max(self.ew_var, 1e-6)),
                self.u3, self.u4 - 3.0)

### The `train()` function

`train()` is called once before any prediction is made. <br />
Its job is to fit whatever model you want and save it to `model_directory_path`.

The baseline does not need training -- the EWMA z-score is computed entirely from the historical segment and the streaming online values, so there is no model to fit. <br />
We simply save a placeholder so `infer()` has something to load.

If you later decide to use a supervized learner (e.g. a gradient boosting model that takes features of the running state and predicts the break probability), this is the function where you would fit and save it.

In [ ]:
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import torch, torch.nn as nn
import gc

# @crunch/keep:on
INFER_PARALLELISM = 8

def train(datasets, model_directory_path):
    import numpy as np, math, os

    datasets = list(datasets)
    rng = np.random.RandomState(42)
    perm = rng.permutation(len(datasets))
    datasets = [datasets[i] for i in perm]
    n_val = len(datasets) // 5
    val, tr = datasets[:n_val], datasets[n_val:]

    def build(ds):
        rows, labels, tcol, sid = [], [], [], []
        for k, (dataset_id, x_hist, x_online, tau) in enumerate(ds):
            sf = StreamingFeatures(np.asarray(x_hist, dtype=np.float64))
            for t, point in enumerate(x_online):
                rows.append(sf.update(float(point)))
                labels.append(1 if (tau is not None and t >= tau) else 0)
                tcol.append(t); sid.append(k)
        return (np.asarray(rows, np.float32), np.asarray(labels, np.int8),
                np.asarray(tcol, np.int32), np.asarray(sid, np.int32))

    cache = "/tmp/feat_cache_pkg_37.npz"
    if os.path.exists(cache):
        d = np.load(cache)
        Xtr, ytr = d["Xtr"], d["ytr"]
        Xva, yva, tva, sva = d["Xva"], d["yva"], d["tva"], d["sva"]
        print("features loaded from cache")
    else:
        Xtr, ytr, _, _ = build(tr)
        Xva, yva, tva, sva = build(val)
        np.savez(cache, Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, tva=tva, sva=sva)
        print("features built")

    m_cat = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                               random_seed=42, verbose=0, allow_writing_files=False,
                               thread_count=8)
    m_cat.fit(Xtr, ytr)
    p_cat = m_cat.predict_proba(Xva)[:, 1]
    del Xtr, ytr
    gc.collect()

    def fit_ar2(z):
        if len(z) < 50:
            return 0.0, 0.0, 1.0
        y = z[2:]; X = np.column_stack([z[1:-1], z[:-2]])
        phi = np.linalg.solve(X.T @ X + 1e-6 * np.eye(2), X.T @ y)
        return float(phi[0]), float(phi[1]), max(float((y - X @ phi).var()), 1e-6)

    def residual_xy(x_hist, x_online, tau):
        h = np.asarray(x_hist, dtype=np.float64)
        o = np.asarray(x_online, dtype=np.float64)
        mu, sd = h.mean(), max(h.std(ddof=1), 1e-9)
        zh = (h - mu) / sd
        p1, p2, rv = fit_ar2(zh)
        z1, z2 = zh[-1], zh[-2]
        n = len(o)
        X = np.empty((n, 8), dtype=np.float32)
        ew = 1.0; a_f = 1.0; a_s = 1.0
        csum = 0.0; csum_min = 0.0; cs_env = 0.0; u_prev = 0.0
        for i, x in enumerate(o):
            z = (x - mu) / sd
            u = (z - p1 * z1 - p2 * z2) / math.sqrt(rv)
            z2, z1 = z1, z
            g = u / math.sqrt(max(ew, 1e-6))
            ew = 0.94 * ew + 0.06 * u * u
            uc = max(min(u, 8), -8); au = min(abs(u), 8)
            a_f = 0.90 * a_f + 0.10 * au; a_s = 0.98 * a_s + 0.02 * au
            lag = uc * u_prev; u_prev = uc
            csum += uc - 0.5
            if csum < csum_min: csum_min = csum
            if csum - csum_min > cs_env: cs_env = csum - csum_min
            cs = cs_env / math.sqrt(i + 1)
            X[i] = (uc, max(min(g, 8), -8), au, uc**2 / 8,
                    min(a_f, 4), min(a_s, 4), max(min(lag, 8), -8), min(max(cs, 0), 8))
        y = np.zeros(n, dtype=np.float32)
        if tau is not None:
            y[tau:] = 1.0
        return X, y

    seq_tr = [residual_xy(h, o, tau) for _, h, o, tau in tr]
    seq_va = [residual_xy(h, o, tau) for _, h, o, tau in val]

    class CausalGRU(nn.Module):
        def __init__(self, nin=8, nh=64):
            super().__init__()
            self.gru = nn.GRU(nin, nh, num_layers=2, batch_first=True, dropout=0.1)
            self.head = nn.Linear(nh, 1)
        def forward(self, x, h=None):
            out, h2 = self.gru(x, h)
            return self.head(out).squeeze(-1), h2

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    bce = nn.BCEWithLogitsLoss(reduction="none")

    def pad_batch(items):
        T = max(len(x) for x, _ in items)
        X = torch.zeros(len(items), T, 8); Y = torch.zeros(len(items), T); M = torch.zeros(len(items), T)
        for j, (x, y) in enumerate(items):
            L = len(x)
            X[j, :L] = torch.from_numpy(x); Y[j, :L] = torch.from_numpy(y); M[j, :L] = 1
        return X.to(dev), Y.to(dev), M.to(dev)

    def pairwise_t_loss(logits, Y, M, margin=0.5, max_pairs=512):
        B, T = logits.shape
        total = logits.sum() * 0.0; n = 0
        ts_ = torch.randperm(T)[:min(T, 48)]
        for t in ts_:
            valid = M[:, t] > 0
            if valid.sum() < 2: continue
            y = Y[valid, t]; l = logits[valid, t]
            pos, neg = l[y > 0.5], l[y < 0.5]
            if len(pos) == 0 or len(neg) == 0: continue
            diff = neg.unsqueeze(0) - pos.unsqueeze(1) + margin
            h = torch.clamp(diff, min=0)
            if h.numel() > max_pairs:
                idx = torch.randperm(h.numel())[:max_pairs]
                h = h.flatten()[idx]
            total = total + h.mean(); n += 1
        return total / max(n, 1)

    # per-t weighted TS-AUC for checkpoint selection
    def eval_ts(pv):
        num = den = 0.0
        for t in np.unique(tva):
            m = tva == t
            npos, nneg = int(yva[m].sum()), int((1 - yva[m]).sum())
            if npos == 0 or nneg == 0:
                continue
            w = npos * nneg
            num += w * roc_auc_score(yva[m], pv[m]); den += w
        return num / den

    def gru_val_preds(model):
        model.eval()
        preds = []
        with torch.no_grad():
            for b in range(0, len(seq_va), 64):
                items = seq_va[b:b + 64]
                X, _, _ = pad_batch(items)
                P = torch.sigmoid(model(X)[0]).cpu().numpy()
                for j, (x, _) in enumerate(items):
                    preds.append(P[j, :len(x)])
        model.train()
        return np.concatenate(preds)

    SEEDS = (42, 123, 777)
    EPOCHS = 48
    states = []
    p_gru_sum = None
    for seed in SEEDS:
        torch.manual_seed(seed); np.random.seed(seed)
        model = CausalGRU().to(dev)
        opt = torch.optim.Adam(model.parameters(), lr=2e-3)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
        order = np.arange(len(seq_tr))
        best_auc, best_state, best_preds = 0.0, None, None
        for epoch in range(EPOCHS):
            np.random.shuffle(order)
            for b in range(0, len(order), 64):
                X, Y, M = pad_batch([seq_tr[i] for i in order[b:b + 64]])
                opt.zero_grad()
                logits = model(X)[0]
                loss = (bce(logits, Y) * M).sum() / M.sum() + 0.5 * pairwise_t_loss(logits, Y, M)
                loss.backward(); opt.step()
            sched.step()
            if epoch % 4 == 3:
                preds = gru_val_preds(model)
                auc = eval_ts(preds)
                if auc > best_auc:
                    best_auc = auc
                    best_preds = preds
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        p_gru_sum = best_preds if p_gru_sum is None else p_gru_sum + best_preds
        states.append(best_state)
        print(f"seed {seed} trained, best solo {best_auc:.4f}")
    p_gru = p_gru_sum / len(SEEDS)

    def blend30(p):
        out = p.copy()
        for k in np.unique(sva):
            idx = np.where(sva == k)[0]
            out[idx] = 0.3 * np.maximum.accumulate(p[idx]) + 0.7 * p[idx]
        return out

    B = 0.35
    print(f"CAT alone  blend30 (val): {eval_ts(blend30(p_cat)):.4f}")
    print(f"STACK b={B} blend30 (val): {eval_ts(blend30((1 - B) * p_cat + B * p_gru)):.4f}")

    joblib.dump({"cat": m_cat, "gru_states": states, "b": B},
                os.path.join(model_directory_path, "model.joblib"))

### The `infer()` function

`infer()` is a **generator**. It must:

1. Load any model saved by `train()`.
2. `yield` once, with no value, to signal readiness to the runner.
3. For each test series `(x_historical, x_online)`:
    - Pre-compute whatever static summaries you need from `x_historical` (it is given in full and cheap to scan once).
    - Loop over the points of `x_online`. After each point, `yield` a `float` in $[0, 1]$ -- **exactly one yield per online point**, in order.

<br />

> **‼ Important**
> ---
> Both the outer `datasets` iterable and each `x_online` can be iterated **only once** in the cloud environment. <br />
> You must produce the score for the current observation before the next one is released.

### The streaming EWMA detector in code

We keep three running scalars per series:

- `mu_ewma`: the EWMA of the online values (tracks the current local mean).
- `n_eff`: the effective sample size of the EWMA (grows as more points arrive, bounded by `1 / (1 - alpha)`).
- `t`: the step index, for optional diagnostics.

At each new observation `x_t` we update these in O(1) and emit the score.

In [ ]:
import torch, torch.nn as nn

def infer(datasets, model_directory_path):
    import numpy as np, math, os

    torch.set_num_threads(1)
    torch.manual_seed(42)

    class CausalGRU(nn.Module):
        def __init__(self, nin=8, nh=64):
            super().__init__()
            self.gru = nn.GRU(nin, nh, num_layers=2, batch_first=True, dropout=0.1)
            self.head = nn.Linear(nh, 1)
        def forward(self, x, h=None):
            out, h2 = self.gru(x, h)
            return self.head(out).squeeze(-1), h2

    bundle = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    cat = bundle["cat"]
    b = bundle["b"]
    grus = []
    for st in bundle["gru_states"]:
        g = CausalGRU()
        g.load_state_dict(st)
        g.eval()
        grus.append(g)

    def fit_ar2(z):
        if len(z) < 50:
            return 0.0, 0.0, 1.0
        y = z[2:]; X = np.column_stack([z[1:-1], z[:-2]])
        phi = np.linalg.solve(X.T @ X + 1e-6 * np.eye(2), X.T @ y)
        return float(phi[0]), float(phi[1]), max(float((y - X @ phi).var()), 1e-6)

    yield  # Signal readiness to the runner.

    for x_historical, x_online in datasets:
        h_arr = np.asarray(x_historical, dtype=np.float64)
        sf = StreamingFeatures(h_arr)
        mu, sd = h_arr.mean(), max(h_arr.std(ddof=1), 1e-9)
        zh = (h_arr - mu) / sd
        p1, p2, rv = fit_ar2(zh)
        z1, z2 = zh[-1], zh[-2]
        ew = 1.0; a_f = 1.0; a_s = 1.0
        csum = 0.0; csum_min = 0.0; cs_env = 0.0; u_prev = 0.0
        hiddens = [None] * len(grus)
        peak = 0.0
        i = 0
        with torch.no_grad():
            for point in x_online:
                f = sf.update(float(point))
                row = np.array([f], dtype=np.float64)
                s_cat = float(cat.predict_proba(row)[0, 1])

                z = (float(point) - mu) / sd
                u = (z - p1 * z1 - p2 * z2) / math.sqrt(rv)
                z2, z1 = z1, z
                g = u / math.sqrt(max(ew, 1e-6))
                ew = 0.94 * ew + 0.06 * u * u
                uc = max(min(u, 8), -8); au = min(abs(u), 8)
                a_f = 0.90 * a_f + 0.10 * au; a_s = 0.98 * a_s + 0.02 * au
                lag = uc * u_prev; u_prev = uc
                csum += uc - 0.5
                if csum < csum_min: csum_min = csum
                if csum - csum_min > cs_env: cs_env = csum - csum_min
                cs = cs_env / math.sqrt(i + 1)
                i += 1
                xin = torch.tensor([[[uc, max(min(g, 8), -8), au, uc**2 / 8,
                                      min(a_f, 4), min(a_s, 4),
                                      max(min(lag, 8), -8), min(max(cs, 0), 8)]]],
                                   dtype=torch.float32)
                s_gru = 0.0
                for gi, gm in enumerate(grus):
                    logit, hiddens[gi] = gm(xin, hiddens[gi])
                    s_gru += float(torch.sigmoid(logit)[0, 0])
                s_gru /= len(grus)

                s = (1 - b) * s_cat + b * s_gru
                if s > peak:
                    peak = s
                yield 0.3 * peak + 0.7 * s

## Local testing

The Crunch CLI ships with a local tester that reproduces the cloud
environment. <br />
It calls `train()` once, then calls `infer()` with the test data, collects the yielded scores, and writes them to `prediction/prediction.parquet`.

This is the same flow the platform will run.

In [ ]:
crunch_tools.test(
    # Uncomment to skip re-training each time
    # force_first_train=False,

    # Uncomment to skip the determinism check
    # no_determinism_check=True,
)

## Previewing the results

In [ ]:
prediction = pd.read_parquet("prediction/prediction.parquet")
prediction.head(10)

prediction
id    time            
10000 3312    0.002722
      3313    0.001283
      3314    0.024189
      3315    0.045730
      3316    0.075652
      3317    0.092703
      3318    0.102657
      3319    0.152654
      3320    0.123343
      3321    0.045093

### Computing TS-AUC locally

Below is a straightforward implementation of the TS-AUC metric: group rows by online time step, compute the AUC cross-sectionally at each step (skipping steps that only have one class), and return the weighted average.

This is exactly what the leaderboard uses, just without the private test set.

In [ ]:
# Load the ground-truth labels supplied with the local tester.
y_test = pd.read_parquet("data/y_test.reduced.parquet")

# Merge predictions with true labels on (id, time).
merged = prediction.merge(
    y_test,
    how="left",
    left_index=True,
    right_index=True,
)

# Add the online step index (0, 1, 2, ...).
merged["time_online"] = merged.groupby("id").cumcount()

# Weighted per-step AUC.
weighted_auc_sum = 0.0
total_weight     = 0.0

for t, group in merged.groupby("time_online"):
    labels = group["target"].values
    scores = group["prediction"].values

    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())
    if n_pos == 0 or n_neg == 0:
        continue

    auc_t  = float(roc_auc_score(labels, scores))
    weight = float(n_pos * n_neg)

    weighted_auc_sum += weight * auc_t
    total_weight     += weight

ts_auc = weighted_auc_sum / total_weight if total_weight > 0 else 0.5
print(f"Local TS-AUC: {ts_auc:.4f}")

Local TS-AUC: 0.5170


## Ideas for stronger solutions

The baseline above is a starting point, not a competitive submission. <br />
Here are directions that typically pay off, in rough order of effort vs. expected benefit:

**Richer streaming statistics.** The EWMA reacts to *mean* shifts but ignores changes in variance, distributional shape, and serial correlation. Try maintaining, in parallel:

- A **CUSUM** of standardized residuals (cumulative deviation from the historical mean), which accumulates evidence over time rather than decaying it.
- Running **variance** and a ratio against $\sigma_H^2$ to catch volatility shifts.
- Fraction of online points beyond $\pm 2\sigma_H$ or $\pm 3\sigma_H$ (tail-mass change).
- Online **autocorrelation** at a few lags, compared to the historical autocorrelation.
- A **likelihood-ratio CUSUM** under a small autoregressive model fit to the historical segment.

All of these can be updated in O(1) per new observation with a bit of care.

**Combine features with a supervized model:**
- If you collect a handful of incremental features like the ones above into a vector, you have a standard tabular classification problem at every online step.
- A gradient boosting model (LightGBM, XGBoost) trained on `(feature_vector, label_t)` pairs from the training set can learn to weight the features much better than a hand-tuned rule.
- Remember to store *all* the state needed to reproduce the feature vector at inference time in O(1).

**Watch your time budget:**
- With 10,000 series and up to 1,000 online steps each, your code may be asked for up to 10 million scores.
- At any step, anything slower than a few microseconds of Python is a liability. Profile!

# Submitting your notebook

To submit your work, you must:

1. Download your notebook (from Colab, Kaggle, or a local copy).
2. Upload it to the competition platform.
3. Create a **run** to validate it against the full test set.

Executing the cell below will take care of everything (only available on Google Colab), or show you how to submit manually.

In [ ]:
# @title  {"display-mode":"form", "form-width":"400px"}

# @markdown Describe your changes, then run the cell.
Message = "" # @param {"type":"string","placeholder":"Short description (optional)"}

# ---
# THIS METHOD IS ONLY POSSIBLE ON COLAB.
# RUNNING THIS CELL WILL PROMPT YOU TO USE THE OLD WAY OF SUBMITTING A NOTEBOOK.

crunch_tools.submit(
    message=Message,
)